# Structure to 11B NMR: frozen diagnostic

Run the released liquid All-nucleus NMRNet checkpoint on existing atom-indexed validation records.

This notebook displays recorded real evaluation results. Model inference is opt-in and never trains parameters.

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'nmr.forward.11B'
example_input = {'smiles': 'CCO', 'formula': 'C2H6O', 'h_nmr': [{'type': '1H NMR', 'chemical_shift': '1.20 (t, 3H), 3.65 (q, 2H)'}], 'c_nmr': [{'type': '13C NMR', 'chemical_shift': '18.2, 58.3'}], 'modality_mask': [1, 1]}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "smiles": "CCO",
  "formula": "C2H6O",
  "h_nmr": [
    {
      "type": "1H NMR",
      "chemical_shift": "1.20 (t, 3H), 3.65 (q, 2H)"
    }
  ],
  "c_nmr": [
    {
      "type": "13C NMR",
      "chemical_shift": "18.2, 58.3"
    }
  ],
  "modality_mask": [
    1,
    1
  ]
}


## Recorded evaluation

Results retain their sample size, conditions and source file; an execution pass does not imply accuracy.

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "nmr.forward.11B",
    "status": "diagnostic_only",
    "model": "NMRNet liquid All-nucleus fold 0",
    "metrics": {
      "mae_ppm": 8.684532299637794,
      "rmse_ppm": 12.436461034675611
    },
    "molecules": 32,
    "atoms": 32,
    "primary_score_eligible": false,
    "limitations": [
      "Nucleus-specific training coverage is unverified.",
      "Training overlap with evaluation records is unaudited.",
      "This model does not condition on solvent.",
      "Atom labels originate from the local prepared pool; experimental assignment provenance is unverified."
    ],
    "label_alignment_verified": true,
    "source_file": "bsi_frozen_diagnostic_20261116.json",
    "sample_limit": 32
  }
]


## Optional frozen inference

Set `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` and configure the workspace and model runtime. Missing weights cause a skip/error, never training.

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
